# Phase 13.4 — Recommendation API Serving

Trace a SIMILAR recommendation from HTTP through the serving service into the existing Phase 11 pipeline.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from fastapi.testclient import TestClient

from productiq.api import create_app
from productiq.serving import PLANNED_ROUTE_RECOMMENDATIONS, RecommendationApiRequest
from productiq.serving.recommendation_service import ProductionRecommendationServingService
from tests.recommendation.test_recommendation_pipeline import _build_catalog, _build_pipeline

## 1. Wire the existing RecommendationPipeline

The HTTP layer never constructs generators or rankers; it receives a ready-made pipeline.

In [ ]:
catalog, filtering = _build_catalog("P1", "P2", "P3", "P4")
pipeline = _build_pipeline(catalog, filtering)
service = ProductionRecommendationServingService(pipeline)
app = create_app(recommendation_service=service)

## 2. POST /api/v1/recommendations (SIMILAR only)

In [ ]:
body = RecommendationApiRequest(seed_product_id="P1", top_k=2).model_dump(mode="json")
with TestClient(app) as client:
    response = client.post(PLANNED_ROUTE_RECOMMENDATIONS, json=body)
response.status_code, response.json()

## 3. Missing seed → 404 (not 500)

Seed resolution failures use the centralized HTTP error mapping.

In [ ]:
with TestClient(app) as client:
    missing = client.post(
        PLANNED_ROUTE_RECOMMENDATIONS,
        json={"seed_product_id": "UNKNOWN", "top_k": 2},
    )
missing.status_code, missing.json()